Read Input File data and convert into Chunks

In [12]:
#2nd Chunking Strategy: Recursive Character Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("employee_leave_policy.pdf")
pages = loader.load()
text = "\n\n".join(page.page_content for page in pages)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
#\n\n -- 

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print(chunk)
    print("Length:", len(chunk))



#chunk1 



--- Chunk 1 ---
Employee Leave Policy  Sample document
Page 1
 EMPLOYEE LEAVE POLICY
Length: 69

--- Chunk 2 ---
Sample Company Policy Document
Policy ID
HR-LEAVE-001
Version
1.0 — Sample
Effective date
Length: 89

--- Chunk 3 ---
Effective date
1 January 2026 (illustrative)
Policy owner
Human Resources
Review cycle
Length: 86

--- Chunk 4 ---
Review cycle
Annually or when applicable law changes
Purpose
Length: 60

--- Chunk 5 ---
This document describes a sample framework for requesting, approving, recording, and administering
Length: 98

--- Chunk 6 ---
and administering employee
Length: 26

--- Chunk 7 ---
leave. It is designed as a realistic example for document-processing and Retrieval-Augmented
Length: 92

--- Chunk 8 ---
Retrieval-Augmented Generation (RAG)
Length: 36

--- Chunk 9 ---
demonstrations.
Important notice
Length: 32

--- Chunk 10 ---
This is a fictional, illustrative policy and is not legal advice or an official company policy.
Length: 95

--- Chunk 11 ---
company 

Converting Chunks into Vectors

In [13]:
import os
import sys


from sentence_transformers import SentenceTransformer

# Load the embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Chunks are already strings; only use .page_content if they are Document objects
texts = [c.page_content if hasattr(c, "page_content") else c for c in chunks]

# Convert all chunks into vectors
vectors = model.encode(texts)

print(vectors)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 724.37it/s]


[[-0.01341252  0.09093408  0.03254393 ...  0.08641397  0.04347856
  -0.01934119]
 [-0.01961714  0.06447425  0.00485145 ... -0.00192476  0.01793732
   0.00446834]
 [-0.05451678  0.02013964 -0.02309616 ... -0.07247847  0.01634225
   0.0596562 ]
 ...
 [-0.04390659  0.04499666 -0.02199919 ... -0.03486433  0.00969412
   0.06221284]
 [-0.04408241  0.06611563  0.02341731 ...  0.01755377 -0.04222295
   0.03390504]
 [-0.00912062  0.09777225  0.04294809 ...  0.02863869  0.08255664
   0.02683826]]


Create Index in Pinecone

Inserting Vectors and Metadata into Pinecone

In [14]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.getenv("PineCone_API_Key"))
name = os.getenv("Pinecone_Index_Name")  # use a new name if the old index still exists

if name not in pc.list_indexes().names():
    pc.create_index(
        name=name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(name)

memory_index_name = os.getenv("Pinecone_Memory_Index_Name", "employeeleavepolicy-longmemory")
if memory_index_name not in pc.list_indexes().names():
    pc.create_index(
        name=memory_index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
memory_index = pc.Index(memory_index_name)

In [ ]:
import os
import time

# Build records: id, embedding, and original text as metadata
records = [
    {
        "id": f"leave-policy-{i}",
        "values": vectors[i].tolist(),
        "metadata": {"text": texts[i], "chunk_index": i},
    }
    for i in range(len(vectors))
]

# Insert into Pinecone in batches
batch_size = 100
for start in range(0, len(records), batch_size):
    batch = records[start:start + batch_size]
    index.upsert(vectors=batch)
    print(f"Inserted {start + len(batch)}/{len(records)}")

time.sleep(5)  # stats can lag briefly
print(index.describe_index_stats())

index.search_records(vector=query_vector, top_k=TOP_K_PER_YEAR)


#

Inserted 100/175
Inserted 175/175
DescribeIndexStatsResponse(dimension=384, total_vector_count=175, metric='cosine', namespaces=1)


Convert User prompt to Vector

In [25]:
from sentence_transformers import SentenceTransformer

# Load the same embedding model used during indexing
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# User question
question = "How many sick leave are available?"



# Convert question into vector
query_vector = model.encode(
    question
).tolist()

print("Query vector dimensions:", len(query_vector))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 980.51it/s]


Query vector dimensions: 384


In [26]:
# Search Pinecone
results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

# Display retrieved chunks
for match in results.matches:
    print("\nScore:", match.score)
    print("Chunk:", match.metadata["text"])
    print("-----------------------------")


Score: 0.73880446
Chunk: balance and approval.
Sick Leave
10 working days per leave
year
-----------------------------

Score: 0.635712
Chunk: 20 working days per leave
year
Planned holidays, personal travel or rest. Subject to
-----------------------------

Score: 0.626040101
Chunk: rather than assume a universal rule.
3.2 Sick Leave
-----------------------------


In [27]:
context = "\n\n".join(
    match.metadata["text"]
    for match in results.matches
)

print("Retrieved Context:")
print(context)

Retrieved Context:
balance and approval.
Sick Leave
10 working days per leave
year

20 working days per leave
year
Planned holidays, personal travel or rest. Subject to

rather than assume a universal rule.
3.2 Sick Leave


In [28]:
import os
import uuid
from dotenv import load_dotenv
import anthropic

load_dotenv()

# Initialize Claude
client = anthropic.Anthropic(api_key=os.getenv("Claude_API_KEY"))

def ask_claude(prompt):
    """Claude's text answer, or None if Claude declined."""
    response = client.beta.messages.create(
        model="claude-opus-5-5",
        max_tokens=16000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",  # if Claude declines, the API retries on a fallback model
        messages=[{"role": "user", "content": prompt}],
    )
    if response.stop_reason == "refusal":
        return None
    return next(b.text for b in response.content if b.type == "text")

MEMORY_THRESHOLD = 0.90       # min cosine similarity to reuse a stored answer



# 1. Check Memory first
mem = memory_index.query(
    vector=query_vector,
    top_k=1,
    include_metadata=True,
)

if mem.matches and mem.matches[0].score >= MEMORY_THRESHOLD:
    hit = mem.matches[0]
    print(f"[Memory HIT] score={hit.score:.3f}, matched question: {hit.metadata['question']}")
    answer = hit.metadata["answer"]
else:
    # 2. Not in memory: usual RAG flow
    print("[Memory MISS] running normal RAG flow")

    prompt = f"""
You are a professional employee policy assistant.

Rephrase the answer to make it clear, natural, and user-friendly.

Important instructions:
- Preserve the original meaning.
- Do not add new information.
- Do not change any numbers or facts.
- Answer directly to the user's question.

User Question:
{question}

Original Answer:
{context}

Rephrased Answer:
"""

    answer = ask_claude(prompt)

    if answer is None:
        answer = "Sorry, I can't answer that question."
    else:
        # 3. Save to memory for next time
        memory_index.upsert(
            vectors=[{
                "id": f"mem-{uuid.uuid4()}",
                "values": query_vector,
                "metadata": {"question": question, "answer": answer},
            }],
        )

print("User Question:", question)
print("\nAI Generated Answer:")
print(answer)

[Memory MISS] running normal RAG flow
User Question: How many sick leave are available?

AI Generated Answer:
You have 10 working days of sick leave available per leave year. Please note that the use of sick leave is subject to your current balance and requires approval.
